In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd

from macro_releases import MacroReleases
from macro_series import MacroSeries

In [2]:
META = {
    'series_id': 'GDP', 'name': 'Gross Domestic Product', 'source': 'BCE',
    'frequency': 'Q', 'unit': '€', 'country': 'Spain', 'category': 'Economy',
}

dates = pd.date_range('2024-01-01', periods=4, freq='QS')
good = pd.Series([2.0, 3.5, np.nan, 10.0], index=dates[[2, 0, 1, 3]])

m = MacroSeries(data=good, **META)
print(m)
print(m.data)

MacroSeries(series_id='GDP', name='Gross Domestic Product', source='BCE', frequency='Q', unit='€', country='Spain', category='Economy', release='latest', mean=np.float64(5.166666666666667), std=np.float64(4.252450274057691), start=Timestamp('2024-01-01 00:00:00'), end=Timestamp('2024-10-01 00:00:00'), n_obs=np.int64(3))
2024-01-01     3.5
2024-04-01     NaN
2024-07-01     2.0
2024-10-01    10.0
dtype: float64


In [3]:
bad_cases = {
    "not a Series": [1, 2, 3],
    "index not dates": pd.Series([2, 3.5, 10]),
    "empty": pd.Series([], index=pd.DatetimeIndex([]), dtype=float),
    "text values": pd.Series(['a', 'b', 'c'], index=dates[:3]),
}
for label, data in bad_cases.items():
    try:
        MacroSeries(data=data, **META)
        print(f"{label}: NO ERROR (wrong)")
    except (TypeError, ValueError) as e:
        print(f"{label}: {type(e).__name__}: {e}")

not a Series: TypeError: data must be a Series, got list
index not dates: TypeError: index must be a pd.DatetimeIndex, got RangeIndex
empty: ValueError: data cannot be empty
text values: ValueError: data values must be numeric, got dtype object


## MacroReleases: all published versions
ALFRED keeps every version of a macro series as it was published. MacroReleases stores them and rebuilds the series as first published, as last revised, or as known on a given date. Data below is made up: January to April, with an extra revision in February.

In [4]:
vintages = pd.DataFrame(
    [
        # date (reference month), release_date (publication), value
        ('2024-01-01', '2024-02-02', 3.7),
        ('2024-01-01', '2024-03-08', 3.8),
        ('2024-02-01', '2024-03-08', 3.9),
        ('2024-02-01', '2024-04-05', 3.9),
        ('2024-02-01', '2024-06-07', 4.0),  # extra revision
        ('2024-03-01', '2024-04-05', 3.8),
        ('2024-03-01', '2024-05-03', 3.7),
        ('2024-04-01', '2024-05-03', 3.9),
        ('2024-04-01', '2024-06-07', 4.0),
    ],
    columns=['date', 'release_date', 'value'],
)
vintages['date'] = pd.to_datetime(vintages['date'])
vintages['release_date'] = pd.to_datetime(vintages['release_date'])
vintages

,date,release_date,value
0,2024-01-01,2024-02-02,3.7
1,2024-01-01,2024-03-08,3.8
2,2024-02-01,2024-03-08,3.9
3,2024-02-01,2024-04-05,3.9
4,2024-02-01,2024-06-07,4.0
5,2024-03-01,2024-04-05,3.8
6,2024-03-01,2024-05-03,3.7
7,2024-04-01,2024-05-03,3.9
8,2024-04-01,2024-06-07,4.0


In [5]:
RELEASES_META = {
    'series_id': 'UNRATE', 'name': "Unemployment Rate", 'source': 'FRED',
    'frequency': 'M', 'unit': '%', 'country': 'US', 'category': 'Labor',
}
releases = MacroReleases(data=vintages, **RELEASES_META)
releases

MacroReleases(series_id='UNRATE', name='Unemployment Rate', source='FRED', frequency='M', unit='%', country='US', category='Labor')

In [6]:
comparison = pd.DataFrame({
    'first': releases.get_release(1).data,
    'second': releases.get_release(2).data,
    'latest': releases.get_release('latest').data,
    'as_of_2024-04-15': releases.get_release('2024-04-15').data,
})
comparison['revision'] = comparison['latest'] - comparison['first']
comparison

,first,second,latest,as_of_2024-04-15,revision
date,,,,,
2024-01-01,3.7,3.8,3.8,3.8,0.1
2024-02-01,3.9,3.9,4.0,3.9,0.1
2024-03-01,3.8,3.7,3.7,3.8,-0.1
2024-04-01,3.9,4.0,4.0,NaN,0.1


In [7]:
releases.get_release(1)

MacroSeries(series_id='UNRATE', name='Unemployment Rate', source='FRED', frequency='M', unit='%', country='US', category='Labor', release=1, mean=np.float64(3.8249999999999997), std=np.float64(0.0957427107756337), start=Timestamp('2024-01-01 00:00:00'), end=Timestamp('2024-04-01 00:00:00'), n_obs=np.int64(4))

In [8]:
bad_releases = {'zero': 0, 'bool': True, 'float': 3.5,
                'unknown_text': 'hola', 'before_any_data': '2020-01-01'}
for label, r in bad_releases.items():
    try:
        releases.get_release(r)
        print(f"{label}: NO ERROR (wrong)")
    except (TypeError, ValueError) as e:
        print(f"{label}: {type(e).__name__}: {e}")

bad_data = {
    'not_a_dataframe': vintages['value'],
    'missing_column': vintages.drop(columns='release_date'),
    'text_values': vintages.assign(value='x'),
    'bad_dates': vintages.assign(release_date='not a date'),
}
for label, d in bad_data.items():
    try:
        MacroReleases(data=d, **RELEASES_META)
        print(f"{label}: NO ERROR (wrong)")
    except (TypeError, ValueError) as e:
        print(f"{label}: {type(e).__name__}: {e}")

zero: ValueError: release must be >= 1 (1 = first publication), got 0
bool: TypeError: release must be an int, 'latest' or a date, got bool
float: TypeError: release must be an int, 'latest' or a date, got float
unknown_text: ValueError: release must be an int, 'latest' or a date, got 'hola'
before_any_data: ValueError: data cannot be empty
not_a_dataframe: TypeError: data must be a DataFrame, got Series
missing_column: ValueError: data is missing columns: ['release_date']
text_values: ValueError: data values must be numeric, got dtype object
bad_dates: ValueError: release_date must contain dates, got dtype object
